# 03 - Prepare the Complete RadGraph-XL NER and RE Dataset

This notebook combines the 300 MIMIC reports and 2,000 Stanford reports,
creates a reproducible source-modality-stratified 70/15/15 report split, and
produces model-ready NER and relation-extraction files.

The original relation direction is preserved. Candidate pairs are ordered, and
the full relation-candidate CSV is written in bounded chunks rather than held
entirely in memory.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import random
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

MIMIC_ZIP = Path(os.environ["RADGRAPH_XL_MIMIC_ZIP"])
STANFORD_JSONL = Path(os.environ["RADGRAPH_XL_STANFORD_JSONL"])
RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME / "interim"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset and split controls. Keep fixed across compared models.
EXPECTED_REPORTS = 2300
RANDOM_SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# Non-gold ordered pairs farther apart than this are excluded. Gold relations
# are retained regardless of distance. Change only as a documented experiment.
RELATION_DISTANCE_THRESHOLD = 32

# Maximum number of relation-candidate rows buffered before appending to CSV.
# This controls RAM usage, not the candidate set or model behaviour.
CANDIDATE_WRITE_CHUNK_SIZE = 100_000

random.seed(RANDOM_SEED)
assert MIMIC_ZIP.exists(), f"MIMIC ZIP not found: {MIMIC_ZIP}"
assert STANFORD_JSONL.exists(), f"Stanford JSONL not found: {STANFORD_JSONL}"

print(
    {
        "run_name": RUN_NAME,
        "output_dir": str(OUTPUT_DIR),
        "relation_distance_threshold": RELATION_DISTANCE_THRESHOLD,
        "candidate_write_chunk_size": CANDIDATE_WRITE_CHUNK_SIZE,
    }
)


In [ ]:
REQUIRED_FIELDS = {"dataset", "doc_key", "sentences", "ner", "relations"}


def normalise_record(record: dict, source: str) -> dict:
    if set(record) != REQUIRED_FIELDS:
        raise ValueError(f"Unexpected fields for {source}: {sorted(record)}")
    dataset = str(record["dataset"])
    source_doc_key = str(record["doc_key"])
    return {
        "source": source,
        "doc_id": f"{dataset}::{source_doc_key}",
        "source_doc_key": source_doc_key,
        "dataset": dataset,
        "tokens": [token for sentence in record["sentences"] for token in sentence],
        "ner": record["ner"],
        "relations": record["relations"],
    }


def load_jsonl_lines(lines, source: str) -> list[dict]:
    records = []
    for raw_line in lines:
        if raw_line.strip():
            records.append(normalise_record(json.loads(raw_line), source))
    return records


def load_mimic_zip(path: Path) -> list[dict]:
    with zipfile.ZipFile(path) as archive:
        members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]
        if len(members) != 1:
            raise ValueError(f"Expected one MIMIC JSONL member, found {members}")
        with archive.open(members[0]) as handle:
            return load_jsonl_lines(handle, "mimic")


def load_stanford_jsonl(path: Path) -> list[dict]:
    with path.open("rb") as handle:
        return load_jsonl_lines(handle, "stanford")


records = load_mimic_zip(MIMIC_ZIP) + load_stanford_jsonl(STANFORD_JSONL)
if len(records) != EXPECTED_REPORTS:
    raise ValueError(f"Expected {EXPECTED_REPORTS} reports, found {len(records)}")

doc_ids = [record["doc_id"] for record in records]
if len(doc_ids) != len(set(doc_ids)):
    raise ValueError("dataset::doc_key is not unique across the combined collection")

report_rows = []
for record in records:
    report_rows.append(
        {
            "source": record["source"],
            "doc_id": record["doc_id"],
            "source_doc_key": record["source_doc_key"],
            "dataset": record["dataset"],
            "token_count": len(record["tokens"]),
            "entity_count": sum(len(sentence_entities) for sentence_entities in record["ner"]),
            "relation_count": sum(len(sentence_relations) for sentence_relations in record["relations"]),
        }
    )

report_df = pd.DataFrame(report_rows)
print(report_df.groupby(["source", "dataset"]).size())
print(
    {
        "reports": len(report_df),
        "tokens": int(report_df["token_count"].sum()),
        "entities": int(report_df["entity_count"].sum()),
        "relations": int(report_df["relation_count"].sum()),
    }
)


In [ ]:
def make_report_splits(frame: pd.DataFrame) -> pd.DataFrame:
    train_df, temp_df = train_test_split(
        frame,
        test_size=TEST_SIZE + VAL_SIZE,
        random_state=RANDOM_SEED,
        stratify=frame["dataset"],
    )
    val_fraction_of_temp = VAL_SIZE / (TEST_SIZE + VAL_SIZE)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=1 - val_fraction_of_temp,
        random_state=RANDOM_SEED,
        stratify=temp_df["dataset"],
    )
    return pd.concat(
        [
            train_df.assign(split="train"),
            val_df.assign(split="validation"),
            test_df.assign(split="test"),
        ],
        ignore_index=True,
    ).sort_values("doc_id").reset_index(drop=True)


split_df = make_report_splits(report_df)
split_by_doc = dict(zip(split_df["doc_id"], split_df["split"]))
split_counts = split_df["split"].value_counts().to_dict()

expected_split_counts = {"train": 1610, "validation": 345, "test": 345}
if split_counts != expected_split_counts:
    raise ValueError(f"Unexpected split counts: {split_counts}")

split_path = OUTPUT_DIR / "report_splits.csv"
split_df.to_csv(split_path, index=False)
print(pd.crosstab(split_df["dataset"], split_df["split"]))
print("Saved:", split_path)


In [ ]:
def parse_entity_label(label: str) -> tuple[str, str]:
    if "::" not in label:
        return label, "unspecified"
    return tuple(label.split("::", 1))


def safe_label(label: str) -> str:
    return label.replace("::", "__").replace(" ", "_").replace("/", "_")


entity_rows = []
relation_rows = []
entities_by_doc: dict[str, list[dict]] = defaultdict(list)
span_to_entity: dict[str, dict[tuple[int, int], str]] = defaultdict(dict)
ambiguous_entity_spans = 0
missing_relation_endpoints = 0

for record in tqdm(records, desc="Extracting entity and relation annotations", unit="report"):
    doc_id = record["doc_id"]
    split = split_by_doc[doc_id]
    dataset = record["dataset"]
    source = record["source"]
    entity_counter = 0

    for sentence_entities in record["ner"]:
        for start, end, label in sentence_entities:
            category, assertion = parse_entity_label(label)
            entity_id = f"{doc_id}::E{entity_counter:04d}"
            entity_counter += 1
            row = {
                "source": source,
                "doc_id": doc_id,
                "dataset": dataset,
                "split": split,
                "entity_id": entity_id,
                "start": int(start),
                "end": int(end),
                "label": label,
                "safe_label": safe_label(label),
                "category": category,
                "assertion": assertion,
            }
            entity_rows.append(row)
            entities_by_doc[doc_id].append(row)
            span = (int(start), int(end))
            if span in span_to_entity[doc_id]:
                ambiguous_entity_spans += 1
            else:
                span_to_entity[doc_id][span] = entity_id

    relation_counter = 0
    for sentence_relations in record["relations"]:
        for head_start, head_end, tail_start, tail_end, relation_label in sentence_relations:
            head_span = (int(head_start), int(head_end))
            tail_span = (int(tail_start), int(tail_end))
            head_entity_id = span_to_entity[doc_id].get(head_span)
            tail_entity_id = span_to_entity[doc_id].get(tail_span)
            if head_entity_id is None or tail_entity_id is None:
                missing_relation_endpoints += 1
                continue
            relation_rows.append(
                {
                    "source": source,
                    "doc_id": doc_id,
                    "dataset": dataset,
                    "split": split,
                    "relation_id": f"{doc_id}::R{relation_counter:04d}",
                    "head_entity_id": head_entity_id,
                    "tail_entity_id": tail_entity_id,
                    "head_start": int(head_start),
                    "head_end": int(head_end),
                    "tail_start": int(tail_start),
                    "tail_end": int(tail_end),
                    "label": relation_label,
                    "safe_label": safe_label(relation_label),
                }
            )
            relation_counter += 1

entities_df = pd.DataFrame(entity_rows)
relations_df = pd.DataFrame(relation_rows)
re_entities_df = (
    entities_df.sort_values(["doc_id", "start", "end", "entity_id"])
    .drop_duplicates(["doc_id", "start", "end"], keep="first")
    .reset_index(drop=True)
)

entities_path = OUTPUT_DIR / "entities.csv"
relations_path = OUTPUT_DIR / "relations.csv"
entities_df.to_csv(entities_path, index=False)
relations_df.to_csv(relations_path, index=False)

print(
    {
        "entities": len(entities_df),
        "unique_entity_spans_for_re": len(re_entities_df),
        "relations": len(relations_df),
        "ambiguous_entity_spans": ambiguous_entity_spans,
        "missing_relation_endpoints": missing_relation_endpoints,
    }
)


In [ ]:
def build_bio_labels(tokens: list[str], entities: list[dict]) -> tuple[list[str], int]:
    labels = ["O"] * len(tokens)
    skipped_overlaps = 0
    for entity in sorted(entities, key=lambda row: (row["start"], row["end"])):
        start = int(entity["start"])
        end = int(entity["end"])
        label = entity["safe_label"]
        if start < 0 or end >= len(tokens) or start > end:
            skipped_overlaps += 1
            continue
        if any(labels[position] != "O" for position in range(start, end + 1)):
            skipped_overlaps += 1
            continue
        labels[start] = f"B-{label}"
        for position in range(start + 1, end + 1):
            labels[position] = f"I-{label}"
    return labels, skipped_overlaps


ner_path = OUTPUT_DIR / "ner_dataset.jsonl"
ner_label_counts = Counter()
overlap_skips = 0

with ner_path.open("w", encoding="utf-8") as handle:
    for record in tqdm(records, desc="Writing NER JSONL", unit="report"):
        doc_id = record["doc_id"]
        labels, skipped = build_bio_labels(record["tokens"], entities_by_doc[doc_id])
        overlap_skips += skipped
        ner_label_counts.update(labels)
        row = {
            "source": record["source"],
            "doc_id": doc_id,
            "dataset": record["dataset"],
            "split": split_by_doc[doc_id],
            "tokens": record["tokens"],
            "bio_labels": labels,
        }
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

print({"ner_jsonl": str(ner_path), "overlap_or_invalid_entities_skipped": overlap_skips})
print("Top BIO labels:", ner_label_counts.most_common(10))


In [ ]:
def span_distance(a_start: int, a_end: int, b_start: int, b_end: int) -> int:
    if a_end < b_start:
        return b_start - a_end
    if b_end < a_start:
        return a_start - b_end
    return 0


gold_relation_label_sets: dict[tuple[str, int, int, int, int], set[str]] = defaultdict(set)
for row in relations_df.itertuples(index=False):
    gold_relation_label_sets[
        (row.doc_id, int(row.head_start), int(row.head_end), int(row.tail_start), int(row.tail_end))
    ].add(row.safe_label)

ambiguous_relation_keys = {key for key, labels in gold_relation_label_sets.items() if len(labels) > 1}
gold_relation_lookup = {
    key: next(iter(labels))
    for key, labels in gold_relation_label_sets.items()
    if key not in ambiguous_relation_keys
}

candidate_columns = [
    "source",
    "doc_id",
    "dataset",
    "split",
    "head_entity_id",
    "tail_entity_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
    "head_label",
    "tail_label",
    "distance",
    "is_gold",
    "label",
]

candidate_path = OUTPUT_DIR / "re_candidate_pool.csv"
temporary_candidate_path = OUTPUT_DIR / "re_candidate_pool.csv.tmp"
candidate_buffer = []
candidate_count = 0
positive_candidate_count = 0
candidate_label_counts = Counter()


def flush_candidate_buffer(writer: csv.DictWriter) -> None:
    if candidate_buffer:
        writer.writerows(candidate_buffer)
        candidate_buffer.clear()


with temporary_candidate_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=candidate_columns)
    writer.writeheader()

    grouped_entities = re_entities_df.groupby("doc_id", sort=False)
    for doc_id, doc_entities in tqdm(grouped_entities, total=re_entities_df["doc_id"].nunique(), desc="Writing ordered RE candidates", unit="report"):
        document_entities = doc_entities.sort_values(["start", "end", "entity_id"]).to_dict(orient="records")
        for head in document_entities:
            for tail in document_entities:
                if head["entity_id"] == tail["entity_id"]:
                    continue
                distance = span_distance(head["start"], head["end"], tail["start"], tail["end"])
                key = (doc_id, head["start"], head["end"], tail["start"], tail["end"])
                if key in ambiguous_relation_keys:
                    continue
                relation_label = gold_relation_lookup.get(key)
                is_gold = relation_label is not None
                if not is_gold and distance > RELATION_DISTANCE_THRESHOLD:
                    continue

                label = relation_label or "no_relation"
                candidate_buffer.append(
                    {
                        "source": head["source"],
                        "doc_id": doc_id,
                        "dataset": head["dataset"],
                        "split": head["split"],
                        "head_entity_id": head["entity_id"],
                        "tail_entity_id": tail["entity_id"],
                        "head_start": head["start"],
                        "head_end": head["end"],
                        "tail_start": tail["start"],
                        "tail_end": tail["end"],
                        "head_label": head["safe_label"],
                        "tail_label": tail["safe_label"],
                        "distance": distance,
                        "is_gold": is_gold,
                        "label": label,
                    }
                )
                candidate_count += 1
                positive_candidate_count += int(is_gold)
                candidate_label_counts[(head["split"], label)] += 1

                if len(candidate_buffer) >= CANDIDATE_WRITE_CHUNK_SIZE:
                    flush_candidate_buffer(writer)

    flush_candidate_buffer(writer)

temporary_candidate_path.replace(candidate_path)
negative_candidate_count = candidate_count - positive_candidate_count

if RELATION_DISTANCE_THRESHOLD == 32 and candidate_count != 5_365_703:
    raise ValueError(f"Unexpected full-dataset candidate count: {candidate_count}")

print(
    {
        "candidate_pairs": candidate_count,
        "positive_pairs": positive_candidate_count,
        "negative_pairs": negative_candidate_count,
        "threshold": RELATION_DISTANCE_THRESHOLD,
        "unique_gold_relation_pairs": len(gold_relation_label_sets),
        "ambiguous_multi_label_relation_pairs_excluded": len(ambiguous_relation_keys),
        "candidate_csv_size_mb": round(candidate_path.stat().st_size / (1024 ** 2), 2),
    }
)
print(pd.Series(candidate_label_counts).unstack(fill_value=0))


In [ ]:
ner_labels = sorted(ner_label_counts)
preferred_relation_order = ["modify", "located_at", "suggestive_of", "no_relation"]
available_relation_labels = {label for _, label in candidate_label_counts}
relation_labels = [label for label in preferred_relation_order if label in available_relation_labels]
relation_labels += sorted(available_relation_labels - set(preferred_relation_order))

label_maps = {
    "ner_label_to_id": {label: index for index, label in enumerate(ner_labels)},
    "ner_id_to_label": {str(index): label for index, label in enumerate(ner_labels)},
    "relation_label_to_id": {label: index for index, label in enumerate(relation_labels)},
    "relation_id_to_label": {str(index): label for index, label in enumerate(relation_labels)},
    "original_entity_label_to_safe_label": {
        original: safe_label(original) for original in sorted(entities_df["label"].unique())
    },
    "original_relation_label_to_safe_label": {
        original: safe_label(original) for original in sorted(relations_df["label"].unique())
    },
}

summary = {
    "run_name": RUN_NAME,
    "dataset_scope": "complete RadGraph-XL collection",
    "random_seed": RANDOM_SEED,
    "relation_distance_threshold": RELATION_DISTANCE_THRESHOLD,
    "reports": int(len(report_df)),
    "reports_by_source": report_df["source"].value_counts().to_dict(),
    "splits": split_counts,
    "datasets_by_split": pd.crosstab(split_df["dataset"], split_df["split"]).to_dict(),
    "entities": int(len(entities_df)),
    "relations": int(len(relations_df)),
    "unique_entity_spans_for_re": int(len(re_entities_df)),
    "ambiguous_entity_spans": int(ambiguous_entity_spans),
    "unique_gold_relation_pairs": int(len(gold_relation_label_sets)),
    "ambiguous_multi_label_relation_pairs_excluded": int(len(ambiguous_relation_keys)),
    "candidate_pairs": int(candidate_count),
    "positive_candidate_pairs": int(positive_candidate_count),
    "negative_candidate_pairs": int(negative_candidate_count),
    "ner_label_count": int(len(ner_labels)),
    "relation_label_count": int(len(relation_labels)),
    "overlap_or_invalid_entities_skipped": int(overlap_skips),
    "missing_relation_endpoints": int(missing_relation_endpoints),
}

label_maps_path = OUTPUT_DIR / "label_maps.json"
summary_path = OUTPUT_DIR / "prep_summary.json"
label_maps_path.write_text(json.dumps(label_maps, indent=2), encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("Wrote:")
for path in [split_path, entities_path, relations_path, ner_path, candidate_path, label_maps_path, summary_path]:
    print(f" - {path} ({path.stat().st_size / (1024 ** 2):.2f} MB)")
